# DL_T11_3C: Unsupervised Anomaly Detection in Motor Bearing Vibration (CWRU)
**MGIT Hyderabad | CM701PC Deep Learning | Task 3, Variant 3C | Team 11**
Team: 23261A6649, 23261A6644, 23261A6622

**Goal:** learn what *normal* bearing vibration looks like using only normal data, then flag departures from it.
Fault labels are used **only** to tune the threshold (validation split) and to test (test split). Nothing labelled as a fault is ever used for training.

**How to run:** Runtime > Run all. Every number in the report must come from the tables and plots this notebook saves in `/content/outputs`.

Sections: 0 Setup, 1 Data download and inspection, 2 Leak-free splits, 3 Models, 4 Experiments, 5 Final models, 6 Thresholds and cost, 7 Plots, 8 Target check and export.

## 0. Setup and configuration
Fixed seeds and one configuration block, so anyone can reproduce the run.

In [ ]:
import os, time, json, random, shutil, urllib.request, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import scipy.io as sio
from scipy.stats import kurtosis, chi2
from sklearn.metrics import precision_recall_curve, average_precision_score, roc_auc_score
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
warnings.filterwarnings("ignore")

SEED = 42
FS = 12_000            # CWRU sampling rate we use (12 kHz drive-end data)
W = 1024               # window length in samples (about 85 ms)
TRAIN_STRIDE = 512     # 50% overlap, training windows only
EVAL_STRIDE = 1024     # no overlap for validation/test windows
VAL_LOADS = (0, 1)     # fault files recorded at motor load 0 and 1 HP -> threshold tuning
TEST_LOADS = (2, 3)    # fault files recorded at load 2 and 3 HP -> final test only
EPOCHS = int(os.environ.get("EPOCHS", 60))
BATCH = 128
N_SEEDS = int(os.environ.get("N_SEEDS", 3))
C_MISS, C_FA = 400_000, 2_000   # Rs per missed fault, Rs per false alarm
MAX_READINGS = 3                # delay rule if data are sampled slower than once a minute
MAX_DELAY_S = 5 * 60            # delay rule for fast-sampled data
DATA_DIR = os.environ.get("CWRU_DIR", "/content/cwru")
OUT_DIR = os.environ.get("OUT_DIR", "/content/outputs")
os.makedirs(DATA_DIR, exist_ok=True); os.makedirs(OUT_DIR, exist_ok=True)

def set_seed(s):
    random.seed(s); np.random.seed(s); tf.keras.utils.set_random_seed(s)
set_seed(SEED)
print("TensorFlow", tf.__version__, "| GPUs:", len(tf.config.list_physical_devices("GPU")))

## 1. Load and inspect the CWRU data
CWRU is a set of `.mat` files, one per experiment. Each file holds a vibration signal measured at the motor's drive end (key ending in `DE_time`).
We use the **12 kHz drive-end** files:
* **Normal baseline:** files 97 to 100 (motor load 0, 1, 2, 3 HP).
* **Faults:** inner race (IR), ball (B), outer race at 6 o'clock (OR), each with fault diameter 0.007, 0.014, 0.021 inch, at loads 0 to 3 HP. That is 36 fault files.

The downloader tries the CWRU server. If it is blocked, it tells you how to upload the files manually (Drive folder or `files.upload`). **Check the file IDs against the CWRU website table before submitting** (see README).

In [ ]:
NORMAL = {0: 97, 1: 98, 2: 99, 3: 100}
_first_id = {("IR","007"):105, ("B","007"):118, ("OR","007"):130,
             ("IR","014"):169, ("B","014"):185, ("OR","014"):197,
             ("IR","021"):209, ("B","021"):222, ("OR","021"):234}
FAULT_FILES = {(t, s, load): first + load for (t, s), first in _first_id.items() for load in range(4)}
URLS = ["https://engineering.case.edu/sites/default/files/{id}.mat",
        "https://csegroups.case.edu/sites/default/files/bearingdatacenter/files/Datafiles/{id}.mat"]

def _valid(path):
    try:
        m = sio.loadmat(path)
        return any(k.endswith("DE_time") for k in m)
    except Exception:
        return False

def fetch(fid):
    path = os.path.join(DATA_DIR, f"{fid}.mat")
    if os.path.exists(path) and _valid(path):
        return path
    for u in URLS:
        try:
            urllib.request.urlretrieve(u.format(id=fid), path)
            if _valid(path):
                return path
        except Exception:
            pass
    raise FileNotFoundError(
        f"Could not download {fid}.mat. Download the 12k drive-end files manually from the CWRU Bearing Data Center "
        f"and put them in {DATA_DIR} (named like 97.mat), e.g. copy them from Google Drive.")

def load_signal(fid):
    m = sio.loadmat(fetch(fid))
    key = [k for k in m if k.endswith("DE_time")][0]
    return m[key].ravel().astype(np.float32)

all_ids = list(NORMAL.values()) + list(FAULT_FILES.values())
signals = {fid: load_signal(fid) for fid in all_ids}
print("Loaded", len(signals), "files")

In [ ]:
# Inspection table: length, duration, mean, std, RMS, kurtosis per file
rows = []
for load, fid in NORMAL.items():
    x = signals[fid]
    rows.append(dict(file=fid, kind="Normal", size="-", load=load, samples=len(x), seconds=len(x)/FS,
                     mean=x.mean(), std=x.std(), rms=np.sqrt((x**2).mean()), kurtosis=kurtosis(x)))
for (t, s, load), fid in FAULT_FILES.items():
    x = signals[fid]
    rows.append(dict(file=fid, kind=t, size=s, load=load, samples=len(x), seconds=len(x)/FS,
                     mean=x.mean(), std=x.std(), rms=np.sqrt((x**2).mean()), kurtosis=kurtosis(x)))
inspect_df = pd.DataFrame(rows)
inspect_df.to_csv(f"{OUT_DIR}/table_data_inspection.csv", index=False)
print(inspect_df.groupby("kind")[["samples", "seconds", "std", "rms", "kurtosis"]].mean().round(3))
print("\nNull/NaN values in any file:", any(np.isnan(x).any() for x in signals.values()))
print("Total normal seconds available:", round(sum(len(signals[f]) for f in NORMAL.values())/FS, 1))

fig, ax = plt.subplots(4, 1, figsize=(10, 7), sharex=True)
for a, (name, fid) in zip(ax, [("Normal (0 HP)", NORMAL[0]), ("Inner race 0.014 in", FAULT_FILES[("IR","014",0)]),
                               ("Ball 0.014 in", FAULT_FILES[("B","014",0)]), ("Outer race 0.014 in", FAULT_FILES[("OR","014",0)])]):
    a.plot(np.arange(2048)/FS, signals[fid][:2048], lw=0.6); a.set_ylabel("accel."); a.set_title(name, fontsize=9)
ax[-1].set_xlabel("time (s)"); plt.tight_layout(); plt.savefig(f"{OUT_DIR}/fig_raw_signals.png", dpi=150); plt.show()

**What to note when you look at this:** the normal recording is only about 8 to 40 seconds per file. This is important later: it limits how well we can *measure* a false-alarm rate per 24 hours. We will say so honestly in the report.

## 2. Leak-free splits
Rules we follow:
1. **Train** = first 60% in time of each normal file. Only normal data.
2. **Validation (normal)** = next 20%, **Test (normal)** = last 20%. A gap of one window is left between segments, so no window overlaps another split.
3. Fault files are never used for training. Faults recorded at loads 0 and 1 are used to **tune the threshold**; faults at loads 2 and 3 are **test only**. So the test faults are different experiments from the tuning faults.
4. The scaler (mean, std) is computed from the training segments only.
5. Models are chosen by *validation* scores, never by test scores.

In [ ]:
def windows(x, w, stride):
    n = (len(x) - w) // stride + 1
    idx = np.arange(w)[None, :] + stride * np.arange(n)[:, None]
    return x[idx]

train_seg, val_norm_seg, test_norm_seg = [], {}, {}
for load, fid in NORMAL.items():
    x = signals[fid]; n = len(x); a, b = int(0.6*n), int(0.8*n)
    train_seg.append(x[:a]); val_norm_seg[load] = x[a+W:b]; test_norm_seg[load] = x[b+W:]

MU = float(np.concatenate(train_seg).mean()); SD = float(np.concatenate(train_seg).std())
norm = lambda x: ((x - MU) / SD).astype(np.float32)

X_train = np.concatenate([windows(norm(s), W, TRAIN_STRIDE) for s in train_seg])[..., None]
val_normal_w = {l: windows(norm(x), W, EVAL_STRIDE)[..., None] for l, x in val_norm_seg.items()}
test_normal_w = {l: windows(norm(x), W, EVAL_STRIDE)[..., None] for l, x in test_norm_seg.items()}
val_fault_w = {k: windows(norm(signals[f]), W, EVAL_STRIDE)[..., None] for k, f in FAULT_FILES.items() if k[2] in VAL_LOADS}
test_fault_w = {k: windows(norm(signals[f]), W, EVAL_STRIDE)[..., None] for k, f in FAULT_FILES.items() if k[2] in TEST_LOADS}
X_val_normal = np.concatenate(list(val_normal_w.values()))

split_table = pd.DataFrame([
    ["Train (normal only)", len(X_train), 0, "normal files 97-100, first 60%, 50% overlap"],
    ["Validation normal", len(X_val_normal), 0, "same files, next 20%"],
    ["Validation fault", sum(len(v) for v in val_fault_w.values()), len(val_fault_w), "fault files at 0 and 1 HP; threshold tuning"],
    ["Test normal", sum(len(v) for v in test_normal_w.values()), 0, "same files, last 20%"],
    ["Test fault", sum(len(v) for v in test_fault_w.values()), len(test_fault_w), "fault files at 2 and 3 HP; final test only"],
], columns=["split", "windows", "fault_files", "note"])
split_table.to_csv(f"{OUT_DIR}/table_splits.csv", index=False)
print(split_table.to_string(index=False))
print("\nScaler from training data only: mean=%.4f std=%.4f" % (MU, SD))
assert not set(val_fault_w) & set(test_fault_w), "validation and test fault files overlap!"
NORMAL_TEST_SECONDS = sum(len(v) for v in test_normal_w.values()) * W / FS
print("Test normal duration: %.1f s" % NORMAL_TEST_SECONDS)

## 3. Models
Both autoencoders take a window of 1024 samples and try to rebuild it. They only ever see normal windows, so they get good at rebuilding normal vibration. A fault looks different, is rebuilt badly, and gets a high **reconstruction error** (mean squared error). That error is the anomaly score.

* **Dense AE (baseline):** fully connected layers; every input sample is treated separately, no notion of "nearby samples".
* **1D CNN AE (improved):** convolutions with stride 2 compress the signal in time; the same small filters slide over the whole window (parameter sharing), so a short repeating shock pattern is captured wherever it appears.

**Bottleneck size** = number of numbers in the code layer. Dense: 1024, 256, 64, 16. CNN: latent length 64 steps x C channels, with C = 16, 4, 1 giving 1024, 256, 64. A bottleneck of 1024 equals the input size (no compression).

In [ ]:
def build_dense(bn):
    h1, h2 = max(512, bn), max(256, bn)    # hidden layers never narrower than the bottleneck
    i = layers.Input((W, 1)); x = layers.Flatten()(i)
    x = layers.Dense(h1, activation="relu")(x); x = layers.Dense(h2, activation="relu")(x)
    z = layers.Dense(bn, name="latent")(x)
    x = layers.Dense(h2, activation="relu")(z); x = layers.Dense(h1, activation="relu")(x)
    x = layers.Dense(W)(x); o = layers.Reshape((W, 1))(x)
    return models.Model(i, o, name=f"dense_bn{bn}")

def build_cnn(c):
    i = layers.Input((W, 1)); x = i
    for f in (16, 32, 64, 64):                       # 1024 -> 64 time steps
        x = layers.Conv1D(f, 7, strides=2, padding="same", activation="relu")(x)
    z = layers.Conv1D(c, 3, padding="same", name="latent")(x)   # latent size = 64 * c
    x = z
    for f in (64, 64, 32, 16):                       # 64 -> 1024 time steps
        x = layers.Conv1DTranspose(f, 7, strides=2, padding="same", activation="relu")(x)
    o = layers.Conv1D(1, 7, padding="same")(x)
    return models.Model(i, o, name=f"cnn_c{c}")

def build(kind, size):
    return build_dense(size) if kind == "dense" else build_cnn(size)

def latent_size(kind, size):
    return size if kind == "dense" else 64 * size

build_dense(64).summary(); build_cnn(1).summary()

## 4. Scoring, thresholds, and event logic
* **Anomaly score** of a window = mean squared reconstruction error.
* **Alarm** if score > threshold. Optionally require `k` alarms in a row (k = 1, 2, 3) to suppress one-off spikes.
* **False alarm episode** = a run of consecutive alarms on normal data (counted once).
* **Detection delay** = number of readings (windows) after the fault starts until the first alarm; in seconds = readings x 1024/12000.
* **Readings:** our readings are 85 ms windows, which is far faster than once a minute, so the "5 minutes" rule applies. We also check the stricter "within 3 readings" rule.
* **False alarms per 24 h** is extrapolated from the normal duration we have. With so few seconds of normal data this is only a rough estimate; we also give a 95% upper bound.

In [ ]:
def recon_error(model, Xw):
    R = model.predict(Xw, batch_size=512, verbose=0)
    return np.mean((Xw - R) ** 2, axis=(1, 2))

def score_all(model):
    return dict(vn={l: recon_error(model, x) for l, x in val_normal_w.items()},
                tn={l: recon_error(model, x) for l, x in test_normal_w.items()},
                vf={k: recon_error(model, x) for k, x in val_fault_w.items()},
                tf={k: recon_error(model, x) for k, x in test_fault_w.items()})

pool = lambda d: np.concatenate(list(d.values()))

def grid_from(sn, sf, n=2001):
    return np.unique(np.quantile(np.concatenate([sn, sf]), np.linspace(0, 1, n)))

def counts(sn, sf, thr_grid):
    a, b = np.sort(sn), np.sort(sf)
    fp = len(a) - np.searchsorted(a, thr_grid, side="right")
    tp = len(b) - np.searchsorted(b, thr_grid, side="right")
    return tp, fp

def best_f1_threshold(sn, sf):
    g = grid_from(sn, sf); tp, fp = counts(sn, sf, g)
    prec = tp / np.maximum(tp + fp, 1); rec = tp / len(sf)
    f1 = 2 * prec * rec / np.maximum(prec + rec, 1e-12)
    tied = np.flatnonzero(f1 >= f1.max() - 1e-12)
    return float(g[tied[len(tied) // 2]]), float(f1.max())      # middle of a tied range = safer margin

def consec(alarm, k):
    if k == 1: return alarm
    return np.convolve(alarm.astype(int), np.ones(k, int))[:len(alarm)] >= k

def episodes(a):
    return int(np.sum(np.diff(np.r_[0, a.astype(int)]) == 1))

def op_counts(S, thr, k, split):
    nrm, flt = (S["vn"], S["vf"]) if split == "val" else (S["tn"], S["tf"])
    fa = sum(episodes(consec(s > thr, k)) for s in nrm.values())
    secs = sum(len(s) for s in nrm.values()) * W / FS
    delays = {}
    for key, s in flt.items():
        idx = np.flatnonzero(consec(s > thr, k))
        delays[key] = int(idx[0]) + 1 if len(idx) else None
    return fa, secs, delays

def fa_upper95(fa, secs):
    return chi2.ppf(0.975, 2 * (fa + 1)) / 2 / secs * 86400

def evaluate_op(S, thr, k, split="test"):
    fa, secs, delays = op_counts(S, thr, k, split)
    flt = S["tf"] if split == "test" else S["vf"]; nrm = S["tn"] if split == "test" else S["vn"]
    sf, sn = pool(flt), pool(nrm)
    tp, fp = int((sf > thr).sum()), int((sn > thr).sum())
    ev = pd.DataFrame([dict(type=key[0], size_in=key[1], load=key[2], windows=len(flt[key]),
                            frac_windows_flagged=float((flt[key] > thr).mean()),
                            delay_readings=d, delay_s=(d * W / FS if d else np.nan),
                            detected_within_3=(d is not None and d <= MAX_READINGS),
                            detected_within_5min=(d is not None and d * W / FS <= MAX_DELAY_S)) for key, d in delays.items()])
    return dict(thr=thr, k=k, window_recall=tp / len(sf), window_precision=tp / max(tp + fp, 1),
                window_fpr=fp / len(sn), fa_episodes=fa, normal_seconds=secs, fa_per_24h=fa / secs * 86400,
                fa_per_24h_upper95=fa_upper95(fa, secs), events_detected_3=int(ev.detected_within_3.sum()),
                events_detected_5min=int(ev.detected_within_5min.sum()), n_events=len(ev),
                mean_delay_readings=float(ev.delay_readings.dropna().mean()) if ev.delay_readings.notna().any() else np.nan,
                missed_events=int((~ev.detected_within_3).sum()), events=ev)

def cost_of(res):
    return C_MISS * res["missed_events"] + C_FA * res["fa_per_24h"]

def select_cost_optimal(S):
    """Search threshold x k on VALIDATION data; minimise Rs 4 lakh x missed events + Rs 2000 x false alarms per 24 h."""
    g = grid_from(pool(S["vn"]), pool(S["vf"]), 801); rows = []
    for k in (1, 2, 3):
        for thr in g:
            fa, secs, d = op_counts(S, thr, k, "val")
            miss = sum(1 for v in d.values() if v is None or v > MAX_READINGS)
            fa24 = fa / secs * 86400
            rows.append((k, float(thr), miss, fa24, C_MISS * miss + C_FA * fa24))
    cdf = pd.DataFrame(rows, columns=["k", "thr", "missed_events", "fa_per_24h", "cost"])
    best = cdf[cdf.cost <= cdf.cost.min() + 1e-9]
    kbest = int(best.k.min()); bk = best[best.k == kbest].sort_values("thr")
    return float(bk.thr.iloc[len(bk) // 2]), kbest, cdf        # middle of the best range

## 5. Experiments: two model families x several bottleneck sizes x 3 seeds
Training uses **normal windows only**. The validation loss for early stopping is also computed on **normal validation windows only** (no fault labels). Fault labels first enter at the point where we compute AUC/AP and tune thresholds on the validation split.
Models are ranked by **validation** average precision. The test numbers are reported but never used to pick a model.

In [ ]:
CONFIGS = [("dense", 1024), ("dense", 256), ("dense", 64), ("dense", 16), ("cnn", 16), ("cnn", 4), ("cnn", 1)]
runs, store = [], {}
for kind, size in CONFIGS:
    for sd in range(N_SEEDS):
        seed = SEED + sd; set_seed(seed)
        m = build(kind, size)
        m.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss="mse")
        es = callbacks.EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True)
        h = m.fit(X_train, X_train, validation_data=(X_val_normal, X_val_normal), epochs=EPOCHS,
                  batch_size=BATCH, shuffle=True, callbacks=[es], verbose=0)
        t0 = time.perf_counter(); S = score_all(m); dt = time.perf_counter() - t0
        n_eval = sum(len(v) for d in (val_normal_w, test_normal_w, val_fault_w, test_fault_w) for v in d.values())
        vn, vf, tn, tf_ = pool(S["vn"]), pool(S["vf"]), pool(S["tn"]), pool(S["tf"])
        thr, _ = best_f1_threshold(vn, vf)
        yv = np.r_[np.zeros(len(vn)), np.ones(len(vf))]; yt = np.r_[np.zeros(len(tn)), np.ones(len(tf_))]
        r = evaluate_op(S, thr, 1, "test")
        runs.append(dict(model=f"{kind}", bottleneck_param=size, latent=latent_size(kind, size), seed=seed,
                         params=m.count_params(), epochs_run=len(h.history["loss"]),
                         train_mse=min(h.history["loss"]), val_normal_mse=min(h.history["val_loss"]),
                         val_auc=roc_auc_score(yv, np.r_[vn, vf]), val_ap=average_precision_score(yv, np.r_[vn, vf]),
                         test_auc=roc_auc_score(yt, np.r_[tn, tf_]), test_ap=average_precision_score(yt, np.r_[tn, tf_]),
                         test_recall_at_valF1thr=r["window_recall"], test_fpr_at_valF1thr=r["window_fpr"],
                         test_sep_ratio=float(np.median(tf_) / np.median(tn)), ms_per_window=dt / n_eval * 1000))
        store[(kind, size, seed)] = (m, h, S)
        print(f"{kind:5s} latent={latent_size(kind,size):5d} seed={seed}  val_AP={runs[-1]['val_ap']:.4f}  test_AUC={runs[-1]['test_auc']:.4f}  epochs={runs[-1]['epochs_run']}")

runs_df = pd.DataFrame(runs); runs_df.to_csv(f"{OUT_DIR}/table_all_runs.csv", index=False)
agg = runs_df.groupby(["model", "latent"]).agg(
    params=("params", "first"), val_ap=("val_ap", "mean"), test_auc_mean=("test_auc", "mean"), test_auc_std=("test_auc", "std"),
    test_ap_mean=("test_ap", "mean"), test_recall=("test_recall_at_valF1thr", "mean"), test_fpr=("test_fpr_at_valF1thr", "mean"),
    sep_ratio=("test_sep_ratio", "mean"), ms_per_window=("ms_per_window", "mean"), train_mse=("train_mse", "mean"),
    val_normal_mse=("val_normal_mse", "mean")).reset_index().sort_values(["model", "latent"], ascending=[False, False])
agg.to_csv(f"{OUT_DIR}/table_model_comparison.csv", index=False)
print("\n=== Model and bottleneck comparison (mean over seeds) ===")
print(agg.round(4).to_string(index=False))

**How to read this table (for Question 1 and 2 of the report):**
* `sep_ratio` = median fault score / median normal score on the test split. Near 1 means the model rebuilds faults as well as normal data, i.e. it cannot tell them apart. This is what an over-wide bottleneck risks.
* `train_mse` very small *and* `sep_ratio` near 1 for the 1024 bottleneck is the signature of "it learned to copy".
* Compare the CNN and the dense model **at the same latent size** (64 vs 64, 256 vs 256) so that the comparison is fair.

In [ ]:
# Sanity baseline with NO model: score = mean squared signal value (window energy). If this alone separates the classes,
# the benchmark is easy and the autoencoders' advantage must be judged by margin, not by AUC.
_e = lambda d: np.concatenate([np.mean(v ** 2, axis=(1, 2)) for v in d.values()])
base_rows = []
for _split, (_n, _f) in {"val": (val_normal_w, val_fault_w), "test": (test_normal_w, test_fault_w)}.items():
    _sn, _sf = _e(_n), _e(_f); _y = np.r_[np.zeros(len(_sn)), np.ones(len(_sf))]
    base_rows.append(dict(split=_split, energy_only_auc=roc_auc_score(_y, np.r_[_sn, _sf]),
                          max_normal_energy=_sn.max(), min_fault_energy=_sf.min()))
baseline_df = pd.DataFrame(base_rows); baseline_df.to_csv(f"{OUT_DIR}/table_energy_baseline.csv", index=False)
print(baseline_df.round(6).to_string(index=False))

In [ ]:
# Choose the final model of each family using VALIDATION average precision (ties go to the smaller latent size)
def pick(kind):
    sub = agg[agg.model == kind].copy(); sub["val_ap_r"] = sub.val_ap.round(4)
    row = sub.sort_values(["val_ap_r", "latent"], ascending=[False, True]).iloc[0]
    size = row.latent if kind == "dense" else int(row.latent // 64)
    return kind, int(size)

FINALS = {"Dense AE (baseline)": pick("dense"), "1D CNN AE (improved)": pick("cnn")}
final = {}
for name, (kind, size) in FINALS.items():
    m, h, S = store[(kind, size, SEED)]
    final[name] = dict(model=m, hist=h, S=S, kind=kind, size=size)
    print(f"{name}: {kind} bottleneck param {size} (latent {latent_size(kind,size)}), selected on validation AP")

## 6. Threshold selection and cost analysis
Two operating points per model, both chosen on **validation** data only:
* **A: best-F1 threshold** (balances precision and recall, ignores costs). Uses k = 1.
* **B: cost-optimal threshold** (and k). Cost = Rs 4,00,000 x missed fault events + Rs 2,000 x false alarms per 24 h.

**Assumption (state this in the report):** the cost compares one 24-hour period in which every fault event in the set occurs once. A missed event = not alarmed within 3 readings. Because we only have a few seconds of normal data, one false alarm episode extrapolates to a very large number per day. This makes the cost curve very sensitive to false alarms, which is a data limitation, not a property of the model.

In [ ]:
op_rows, events_tables = [], {}
for name, d in final.items():
    S = d["S"]; vn, vf = pool(S["vn"]), pool(S["vf"])
    thrA, f1A = best_f1_threshold(vn, vf)
    thrB, kB, cdf = select_cost_optimal(S)
    d.update(thrA=thrA, thrB=thrB, kB=kB, cost_df=cdf)
    for label, thr, k in (("A: best-F1", thrA, 1), ("B: cost-optimal", thrB, kB)):
        for split in ("val", "test"):
            r = evaluate_op(S, thr, k, split)
            ev = r.pop("events"); events_tables[(name, label, split)] = ev
            r["cost_rs"] = cost_of(r)
            op_rows.append(dict(model=name, operating_point=label, split=split, **r))
ops_df = pd.DataFrame(op_rows)
cols = ["model", "operating_point", "split", "thr", "k", "window_recall", "window_precision", "window_fpr", "fa_episodes",
        "fa_per_24h", "fa_per_24h_upper95", "events_detected_3", "events_detected_5min", "n_events", "mean_delay_readings", "cost_rs"]
ops_df[cols].to_csv(f"{OUT_DIR}/table_operating_points.csv", index=False)
print(ops_df[ops_df.split == "test"][cols].round(4).to_string(index=False))

ev_all = pd.concat([ev.assign(model=m_, operating_point=o_) for (m_, o_, s_), ev in events_tables.items() if s_ == "test"])
ev_all.to_csv(f"{OUT_DIR}/table_events_test.csv", index=False)
print("\nPer fault type (test, window-level share of windows flagged):")
print(ev_all.groupby(["model", "operating_point", "type"]).frac_windows_flagged.mean().round(3).to_string())

## 7. Plots
Each plot is saved as a PNG in the outputs folder for the report.

In [ ]:
colors = {"Dense AE (baseline)": "tab:orange", "1D CNN AE (improved)": "tab:blue"}

# 7a. Training curves of the final models
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
for a, (name, d) in zip(ax, final.items()):
    a.plot(d["hist"].history["loss"], label="train (normal)"); a.plot(d["hist"].history["val_loss"], label="validation (normal)")
    a.set_yscale("log"); a.set_title(name, fontsize=9); a.set_xlabel("epoch"); a.set_ylabel("MSE"); a.legend()
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/fig_training_curves.png", dpi=150); plt.show()

# 7b. Reconstruction examples (test normal vs test fault)
w_norm = test_normal_w[2][5:6]; w_fault = test_fault_w[("OR", "021", 2)][5:6]
fig, ax = plt.subplots(2, 2, figsize=(11, 5), sharex=True)
for j, (name, d) in enumerate(final.items()):
    for i, (lab, w) in enumerate((("normal", w_norm), ("outer-race fault", w_fault))):
        r = d["model"].predict(w, verbose=0)
        ax[i, j].plot(w[0, :, 0], lw=0.7, label="input"); ax[i, j].plot(r[0, :, 0], lw=0.7, label="reconstruction")
        ax[i, j].set_title(f"{name} | {lab} | MSE={np.mean((w-r)**2):.3f}", fontsize=8)
ax[0, 0].legend(fontsize=7); plt.tight_layout(); plt.savefig(f"{OUT_DIR}/fig_reconstruction_examples.png", dpi=150); plt.show()

# 7c. Reconstruction error distributions on the test split with both thresholds
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
for a, (name, d) in zip(ax, final.items()):
    sn, sf = pool(d["S"]["tn"]), pool(d["S"]["tf"])
    bins = np.logspace(np.log10(min(sn.min(), sf.min())), np.log10(max(sn.max(), sf.max())), 60)
    a.hist(sn, bins=bins, alpha=0.6, label="normal"); a.hist(sf, bins=bins, alpha=0.6, label="fault"); a.set_xscale("log")
    a.axvline(d["thrA"], color="g", ls="--", label="A: best F1"); a.axvline(d["thrB"], color="r", ls=":", label="B: cost-optimal")
    a.set_title(name, fontsize=9); a.set_xlabel("reconstruction error (log)"); a.set_ylabel("windows"); a.legend(fontsize=7)
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/fig_error_histograms.png", dpi=150); plt.show()

# 7d. Constructed stream: test normal (load 2) followed by three fault files (error vs reading index)
fig, ax = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
keys = [("IR", "014", 2), ("B", "014", 2), ("OR", "021", 2)]
for a, (name, d) in zip(ax, final.items()):
    seq = [d["S"]["tn"][2]] + [d["S"]["tf"][k] for k in keys]; xs = np.concatenate(seq)
    a.semilogy(xs, lw=0.7, color=colors[name]); edge = 0
    for s_, lab in zip(seq, ["normal"] + [f"{k[0]} {k[1]}in" for k in keys]):
        a.axvline(edge, color="gray", lw=0.5); a.text(edge + 1, a.get_ylim()[1], lab, fontsize=7, va="top"); edge += len(s_)
    a.axhline(d["thrA"], color="g", ls="--", label="A: best F1"); a.axhline(d["thrB"], color="r", ls=":", label="B: cost-optimal")
    a.set_ylabel("error"); a.set_title(name + " (segments joined for display; the fault starts at each grey line)", fontsize=8); a.legend(fontsize=7)
ax[-1].set_xlabel("reading index (1 reading = 1024 samples = 85 ms)")
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/fig_error_stream.png", dpi=150); plt.show()

# 7e. Precision-recall curves on test, with both thresholds marked
PI = 0.001   # assumed share of faulty readings in real operation (for the adjusted panel)
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for name, d in final.items():
    sn, sf = pool(d["S"]["tn"]), pool(d["S"]["tf"]); y = np.r_[np.zeros(len(sn)), np.ones(len(sf))]; s = np.r_[sn, sf]
    p, r, _ = precision_recall_curve(y, s)
    ax[0].plot(r, p, color=colors[name], label=f"{name} (AP={average_precision_score(y, s):.3f})")
    g = grid_from(sn, sf, 1001); tp, fp = counts(sn, sf, g); tpr, fpr = tp / len(sf), fp / len(sn)
    ax[1].plot(tpr, tpr * PI / np.maximum(tpr * PI + fpr * (1 - PI), 1e-12), color=colors[name], label=name)
    for lab, thr, mk in (("A", d["thrA"], "o"), ("B", d["thrB"], "s")):
        tp_, fp_ = (sf > thr).sum(), (sn > thr).sum(); rec_, pre_ = tp_ / len(sf), tp_ / max(tp_ + fp_, 1)
        ax[0].scatter(rec_, pre_, marker=mk, s=60, color=colors[name], edgecolor="k", zorder=5)
        ax[1].scatter(rec_, rec_ * PI / max(rec_ * PI + (fp_ / len(sn)) * (1 - PI), 1e-12), marker=mk, s=60, color=colors[name], edgecolor="k", zorder=5)
ax[0].axvline(0.85, color="gray", ls=":"); ax[0].set_title("Precision-recall (test; circle = A best-F1, square = B cost-optimal)", fontsize=8)
ax[1].axvline(0.85, color="gray", ls=":"); ax[1].set_title(f"Same curve, precision adjusted to {PI*100:.1f}% faulty readings", fontsize=8)
for a in ax: a.set_xlabel("recall"); a.set_ylabel("precision"); a.legend(fontsize=7)
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/fig_precision_recall.png", dpi=150); plt.show()

# 7f. Model and bottleneck comparison
fig, ax = plt.subplots(1, 3, figsize=(13, 3.8))
for kind, c in (("dense", "tab:orange"), ("cnn", "tab:blue")):
    sub = runs_df[runs_df.model == kind].groupby("latent")
    mean, std = sub.test_auc.mean(), sub.test_auc.std().fillna(0)
    ax[0].errorbar(mean.index, mean.values, yerr=std.values, marker="o", color=c, label=kind, capsize=3)
    ax[1].plot(sub.test_sep_ratio.mean().index, sub.test_sep_ratio.mean().values, marker="o", color=c, label=kind)
    ax[2].plot(sub.train_mse.mean().index, sub.train_mse.mean().values, marker="o", color=c, label=kind)
for a, t in zip(ax, ["Test ROC-AUC vs bottleneck", "Median fault / median normal error (test)", "Best training MSE (normal)"]):
    a.set_xscale("log", base=2); a.set_xlabel("latent size (numbers in code layer)"); a.set_title(t, fontsize=9); a.legend(fontsize=7)
ax[1].set_yscale("log"); ax[2].set_yscale("log")
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/fig_model_comparison.png", dpi=150); plt.show()

# 7g. Cost vs threshold (validation) with A and B marked
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
for a, (name, d) in zip(ax, final.items()):
    c = d["cost_df"]; c = c[c.k == d["kB"]]
    a.semilogx(c.thr, c.cost / 1e5, color=colors[name]); a.set_yscale("symlog", linthresh=1)
    a.axvline(d["thrA"], color="g", ls="--", label="A: best F1"); a.axvline(d["thrB"], color="r", ls=":", label="B: cost-optimal")
    a.set_xlabel("threshold"); a.set_ylabel("cost (Rs lakh, validation)"); a.set_title(f"{name}, k={d['kB']}", fontsize=9); a.legend(fontsize=7)
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/fig_cost_curve.png", dpi=150); plt.show()

## 8. Check against every target, and export
The checker prints PASS / FAIL / NOT VERIFIABLE. **Report whatever it says.** If a target is not met, say why in the report.

In [ ]:
def verdict(ok, reliable=True):
    return ("PASS" if ok else "FAIL") if reliable else ("PASS (not statistically verifiable)" if ok else "FAIL")

target_rows = []
for name in final:
    for label in ("A: best-F1", "B: cost-optimal"):
        r = ops_df[(ops_df.model == name) & (ops_df.operating_point == label) & (ops_df.split == "test")].iloc[0]
        target_rows += [
            dict(model=name, op=label, target="Recall on fault events >= 0.85 (window-level)", value=f"{r.window_recall:.3f}", result=verdict(r.window_recall >= 0.85)),
            dict(model=name, op=label, target="Recall on fault events >= 0.85 (event-level, within 3 readings)", value=f"{r.events_detected_3}/{r.n_events}", result=verdict(r.events_detected_3 / r.n_events >= 0.85)),
            dict(model=name, op=label, target="False alarms < 2 per 24 h (point estimate)", value=f"{r.fa_per_24h:.2f} (95% upper bound {r.fa_per_24h_upper95:.0f}; only {r.normal_seconds:.0f} s of normal test data)",
                 result=verdict(r.fa_per_24h < 2, reliable=r.fa_per_24h_upper95 < 2)),
            dict(model=name, op=label, target="Detection delay < 5 min (or within 3 readings)", value=f"{r.events_detected_5min}/{r.n_events} within 5 min; {r.events_detected_3}/{r.n_events} within 3 readings; mean {r.mean_delay_readings:.2f} readings",
                 result=verdict(r.events_detected_5min == r.n_events)),
            dict(model=name, op=label, target="No labelled fault data used for training", value="train windows from normal files 97-100 only", result="PASS (by construction)")]
targets_df = pd.DataFrame(target_rows); targets_df.to_csv(f"{OUT_DIR}/table_targets.csv", index=False)
pd.set_option("display.max_colwidth", 140); print(targets_df.to_string(index=False))

In [ ]:
# Export: summary markdown, settings, AI-use declaration, zip
def md(df):
    try: return df.to_markdown(index=False, floatfmt=".4f")
    except Exception: return df.to_string(index=False)

with open(f"{OUT_DIR}/results_tables.md", "w") as f:
    f.write("# Auto-generated results (copy into the report)\n\n## Splits\n" + md(split_table) +
            "\n\n## Model and bottleneck comparison\n" + md(agg.round(4)) +
            "\n\n## Operating points (test)\n" + md(ops_df[ops_df.split == 'test'][cols].round(4)) +
            "\n\n## Target check\n" + md(targets_df) + "\n")
json.dump(dict(seed=SEED, fs=FS, window=W, train_stride=TRAIN_STRIDE, eval_stride=EVAL_STRIDE, val_loads=VAL_LOADS,
               test_loads=TEST_LOADS, epochs=EPOCHS, batch=BATCH, n_seeds=N_SEEDS, cost_miss=C_MISS, cost_fa=C_FA,
               scaler_mean=MU, scaler_std=SD, finals={k: dict(kind=v["kind"], size=v["size"], thrA=v["thrA"], thrB=v["thrB"], kB=v["kB"]) for k, v in final.items()},
               tf=tf.__version__), open(f"{OUT_DIR}/run_settings.json", "w"), indent=2)
open(f"{OUT_DIR}/AI_usage_declaration.txt", "w").write(
    "AI tool use declaration (Team 11, Task 3, Variant 3C)\n"
    "Tool: Claude (Anthropic) was used as a coding and writing assistant to draft the notebook structure, model code, "
    "evaluation functions, README and report template.\n"
    "Human work: running the notebook, checking the CWRU file IDs against the CWRU website, interpreting every result, "
    "writing the final report text from our own numbers, and preparing to explain all code in the viva.\n"
    "No results in the report were produced by the AI; all numbers come from running this notebook.\n")
shutil.make_archive("/content/DL_T11_3C_outputs", "zip", OUT_DIR)
print("Saved to", OUT_DIR, "and /content/DL_T11_3C_outputs.zip")
try:
    from google.colab import files; files.download("/content/DL_T11_3C_outputs.zip")
except Exception:
    pass